Final wrangling script, wherein we collect all the core ML features needed for downstream analyses.

[Runtime: ~1-5 min, depending on dataset size as well as number & types of features being exported]

---------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
### IMPORTS:
import os, csv, json, glob, re
from pathlib import Path
import shutil
import tempfile
import numpy as np
import pandas as pd

In [ ]:
# ============================
# GLOBAL PARAMETERS
# ============================

# General:
# SUBSET      = config['subset']        # <-- Will ignore all subsetting here, we'll just grab all target files from the manifest, and fill in any missing data with NaNs
HARD_STOP   = config['hard_errors']
RANDOM_SEED = config['random_seed']


### DATA-EXPORTING TOGGLES:
OUTPUT_ANATOMY_MEASURES             = config['output_sets']['morphometry']
OUTPUT_HMM_FEATURES                 = config['output_sets']['HMM_features']
OUTPUT_HMM_STATES                   = config['output_sets']['HMM_states']
OUTPUT_GRAPH_METRICS_SUMMARIES      = config['output_sets']['graph_metrics_un_epoched']
OUTPUT_GRAPH_METRICS_TIME_RESOLVED  = config['output_sets']['graph_metrics_epoched']


# ============================
# SET PATHS
# ============================

BASE_DIRECTORY        = Path(config['root_output_directory'])

RUN_MANIFEST_PATH     = BASE_DIRECTORY / 'subject_manifest.csv'
fMRI_PARAMETERS_PATH  = BASE_DIRECTORY / 'fMRI_manifest.csv'

# Set final output dir:
FINAL_OUTPUT_DIR = config['final_output_dir']

# Set specific output subdirs to wrangle all final, ML-viable data from:

# Anatomy tables:
MORPHOMETRY_DIR = Path(BASE_DIRECTORY) / config['morphometry']['morphometry_output_dir']
MORPHOMETRY_FILE = Path(MORPHOMETRY_DIR) / 'morphometry.csv'    # <-- NOTE: Currently assumes 'EXPORT_MODE' == 'single' (frozen config variable)

# Graph metrics tables:
GRAPH_METRICS_DIR = Path(BASE_DIRECTORY) / config['compute_correlations']['metrics_output_dir']
UN_EPOCHED_DIR    = Path(GRAPH_METRICS_DIR) / 'un-epoched'
EPOCHED_DIR       = Path(GRAPH_METRICS_DIR) / 'epoched'

# HMM-derived features:
HMM_FEATURES_DIR = Path(BASE_DIRECTORY) / config['HMM_decoding']['output_directory_name']
HMM_STATES_DIR   = Path(BASE_DIRECTORY) / config['interpretation_output_dir']


# ============================
### SET OTHER REQUIRED CONFIG.YAML PARAMETERS:
# ============================

# Grab epoching parameters:                 <-- Needed here for grabbing the correct subdir in the 'EPOCHED' output folder:
EPOCH_LENGTH  = config['compute_correlations']['epoch_length']
if EPOCH_LENGTH in (None, 0, 0.0):  # Normalize null/zero to 'None'
    EPOCH_LENGTH = None
else:
    EPOCH_LENGTH = int(EPOCH_LENGTH)
EPOCH_OVERLAP = config['compute_correlations']['epoch_overlap']
if EPOCH_OVERLAP in (None, 0, 0.0):  # Normalize null/zero to 0
    EPOCH_OVERLAP = 0
else:
    EPOCH_OVERLAP = int(EPOCH_OVERLAP)


# Grab dataset-selection config variables:  <-- Needed here for output-subdirectory targeting purposes
DATA_DIR = Path(BASE_DIRECTORY) / config['ML_prep']['training_data_dir']
DATASET_SELECTOR    = str(config["ML_training"]["dataset_selector"]).strip().lower()
DATASET_MANUAL_PATH = config["ML_training"].get("dataset_path", None)


# ============================
# LOAD MANIFESTS
# ============================

RUN_MANIFEST = pd.read_csv(RUN_MANIFEST_PATH)
fMRI_runs    = pd.read_csv(fMRI_PARAMETERS_PATH)

----------

Filter MANIFEST table down to main export columns:

In [ ]:
final_targets = fMRI_runs.copy()[['subject_ID', 'session_ID', 'group_ID', 'protocol_code']]
final_targets.head()

Get name of current target modeling dataset; needed for grabbing the correct subfolders in the HMM-derived feature table directories:

In [ ]:
# ---------------------------------------
# Resolve DATASET_NAME -- needed for subdirectory targeting:
# ---------------------------------------
DATASET_NAME = None

if DATASET_SELECTOR == "latest":
    pointer_path = Path(DATA_DIR) / "LATEST_DATASET.json"
    if not pointer_path.exists():
        raise FileNotFoundError(
            f"[INIT ERROR] dataset_selector='latest' but pointer file not found:\n  {pointer_path}")
    with open(pointer_path, "r") as f:
        pointer = json.load(f)
    # Pointer contains a full dataset_dir, e.g. ".../ML_training_data/<DATASET_NAME>/"
    pointer_dataset_dir = Path(pointer.get("dataset_dir", "")).expanduser()
    if pointer_dataset_dir is None or str(pointer_dataset_dir).strip() == "":
        raise RuntimeError(
            f"[INIT ERROR] Pointer file exists but missing/empty 'dataset_dir':\n  {pointer_path}")
    DATASET_NAME = pointer_dataset_dir.name
elif DATASET_SELECTOR == "manual":
    if DATASET_MANUAL_PATH is None or str(DATASET_MANUAL_PATH).strip() == "":
        raise ValueError(
            "[INIT ERROR] dataset_selector='manual' but ML_training.dataset_path is empty.")
    manual_dir = Path(str(DATASET_MANUAL_PATH)).expanduser()
    DATASET_NAME = manual_dir.name  # the subfolder name identifies the dataset
else:
    raise ValueError(
        f"[INIT ERROR] ML_training.dataset_selector must be 'latest' or 'manual' "
        f"(got: {DATASET_SELECTOR})")

assert DATASET_NAME is not None
print(f"\n  --> Collecting HMM-derived features from dataset subdirectories:  '{DATASET_NAME}'")

--------

Load morphometry / anatomy table (static directory location):

- Loads as 'anatomy_table' -- this should already be in the correct flattened format, so no major data-handling necessary here:

In [ ]:
# Load morphometry table:
anatomy_table = pd.read_csv(MORPHOMETRY_FILE)

# Quick re-ordering of columns to put the 'GLOBAL' set (if present) first:
# Identify columns
id_cols = [column for column in ["subject_ID", "session_ID"] if column in anatomy_table.columns]
global_cols = [
    column for column in anatomy_table.columns
    if "__GLOBAL__" in column and column not in id_cols]
other_cols = [
    column for column in anatomy_table.columns
    if column not in id_cols and column not in global_cols]
anatomy_table = anatomy_table.loc[:, id_cols + global_cols + other_cols]

# Quick sanity-check to make sure all target subject_IDs are present in morphometry table:
missing_in_anatomy = set(final_targets['subject_ID'].unique()) - set(anatomy_table['subject_ID'].unique())
if missing_in_anatomy:
    print(f"[ERROR] {len(missing_in_anatomy)} subject_ID(s) missing from anatomy_table:")
    for sid in sorted(missing_in_anatomy):
        print(f"  - {sid}")
    if HARD_STOP:
        raise ValueError(f"[ERROR] Some subject_IDs in MEG manifest missing from stored morphometry/anatomy table.")
else:
    print("[OK] anatomy_table contains a row for every subject_ID in final_targets.")

Next, we grab the 'UN-EPOCHED' graph metrics summary data. Unlike the morphometry/anatomy table, the target pair of data tables exist as separate files for each subject_ID x session_ID pair, and are NOT pre-flattened; so some significant unpacking and wrangling is required:

- All output columns from this step are prefixed by the 'UN_EPOCHED_DATA_COLUMN_PREFIX' substring, for easy downstream selection/filtering

In [ ]:
# --------------------------------------------------------------------------------
# Build un_epoched_graph_metrics
#       (includes graceful NaN-filling, global ROI padding, & duplicate-checking)
# --------------------------------------------------------------------------------

#########################################################
#########################################################
# Set prefix to append to all output columns for UN-EPOCHED data measures from this cell:
UN_EPOCHED_DATA_COLUMN_PREFIX = "sessionLevel__"        # <-- Should contain hard separator suffix, e.g. '__'
#########################################################
#########################################################


expected_pairs = sorted(set(zip(final_targets["subject_ID"], final_targets["session_ID"])))

centrality_pat = re.compile(r"^(?P<sub>[^_]+)_(?P<ses>[^_]+)_un-epoched_centrality\.csv$")
global_pat     = re.compile(r"^(?P<sub>[^_]+)_(?P<ses>[^_]+)_un-epoched_global_metrics\.csv$")

# Map [subject_ID, session_ID] --> file path:
centrality_files = {}
global_files     = {}

for f in UN_EPOCHED_DIR.glob("*.csv"):
    name = f.name
    m = centrality_pat.match(name)
    if m:
        centrality_files[(m.group("sub"), m.group("ses"))] = f
        continue
    m = global_pat.match(name)
    if m:
        global_files[(m.group("sub"), m.group("ses"))] = f
        continue

# -----------------------------
# Compute GLOBAL ROI padding width across dataset:
# -----------------------------
global_max_roi = None
roi_col_name_seen = None
roi_padding_width = 1

# Only scan centrality tables that correspond to expected pairs (avoids unrelated files):
pairs_with_centrality = [p for p in expected_pairs if p in centrality_files]

for sub, ses in pairs_with_centrality:
    pth = centrality_files[(sub, ses)]
    try:
        # Read header to find ROI label column; then read only that column:
        header_cols = pd.read_csv(pth, nrows=0).columns.tolist()
        roi_cols = [c for c in header_cols if str(c).endswith("_ROI_label")]
        if len(roi_cols) != 1:
            continue
        roi_col = roi_cols[0]
        roi_col_name_seen = roi_col_name_seen or roi_col

        roi_series = pd.read_csv(pth, usecols=[roi_col])[roi_col]
        roi_numeric = pd.to_numeric(roi_series, errors="coerce")
        if roi_numeric.notna().any():
            mx = int(roi_numeric.dropna().max())
            global_max_roi = mx if global_max_roi is None else max(global_max_roi, mx)
    except Exception:
        # Best-effort scan only; ignore failures here (will degrade gracefully later):
        continue

if global_max_roi is not None:
    roi_padding_width = max(1, len(str(int(global_max_roi))))
    print(f"[INFO] Global ROI padding width set to {roi_padding_width} (max ROI label observed = {global_max_roi}).")
else:
    print("[WARN] Could not determine global ROI padding width (no readable ROI labels found). Defaulting to width=1.")
    roi_padding_width = 1


def _flatten_centrality_to_row(centrality_df: pd.DataFrame, subject_id: str, session_id: str, pad: int) -> dict:
    """
    Returns one wide row dict:
      subject_ID, session_ID, ROI-###_<metric>=value ...
    Missing/unparseable values become NaN.
    Raises on malformed schema (caller catches and degrades to IDs-only row).
    """
    roi_cols = [c for c in centrality_df.columns if str(c).endswith("_ROI_label")]
    if len(roi_cols) != 1:
        raise RuntimeError(f"Expected exactly one '*_ROI_label' column, found {len(roi_cols)}: {roi_cols}")
    roi_col = roi_cols[0]

    df = centrality_df.copy()

    # Coerce ROI labels to ints; drop rows where ROI is not numeric:
    roi_numeric = pd.to_numeric(df[roi_col], errors="coerce")
    df = df.loc[roi_numeric.notna()].copy()
    df[roi_col] = roi_numeric.loc[roi_numeric.notna()].astype(int)
    if df.empty:
        return {"subject_ID": subject_id, "session_ID": session_id}
    metric_cols = [c for c in df.columns if c != roi_col]
    if not metric_cols:
        return {"subject_ID": subject_id, "session_ID": session_id}

    row = {"subject_ID": subject_id, "session_ID": session_id}

    for _, r in df.iterrows():
        roi_val = int(r[roi_col])
        roi_token = f"ROI-{roi_val:0{pad}d}"
        for mc in metric_cols:
            row[f"{roi_token}_{mc}"] = pd.to_numeric(r[mc], errors="coerce")

    return row


# -----------------------------
# Build rows (one per expected pair), NaN-fill on missing / unreadable:
# -----------------------------
rows = []
missing_file_pairs = []
parse_fail_pairs = []

for sub, ses in expected_pairs:
    row = {"subject_ID": sub, "session_ID": ses}

    cent_path = centrality_files.get((sub, ses), None)
    glob_path = global_files.get((sub, ses), None)

    if cent_path is None or glob_path is None:
        has_cent = cent_path is not None
        has_glob = glob_path is not None

        print(
            f"[WARN] Missing un-epoched inputs for {sub} | {ses} "
            f"(centrality={'OK' if has_cent else 'MISSING'}, "
            f"global_metrics={'OK' if has_glob else 'MISSING'})")

        missing_file_pairs.append((sub, ses, has_cent, has_glob))
        rows.append(row)  # <-- IDs only; metrics remain NaN
        continue

    try:
        centrality_df = pd.read_csv(cent_path)
        global_df     = pd.read_csv(glob_path)

        # Centrality --> wide format
        row.update(_flatten_centrality_to_row(centrality_df, subject_id=sub, session_id=ses, pad=roi_padding_width))

        # Global metrics --> append (uses first row if present; else leaves NaNs)
        if global_df.shape[0] >= 1:
            g = global_df.iloc[0].to_dict()
            g.pop("subject_ID", None)
            g.pop("session_ID", None)
            for k, v in g.items():
                row[k] = pd.to_numeric(v, errors="coerce")

    except Exception as e:
        parse_fail_pairs.append((sub, ses, str(e)))
        row = {"subject_ID": sub, "session_ID": ses}

    rows.append(row)

un_epoched_graph_metrics = pd.DataFrame(rows)

# Stable sorting:
un_epoched_graph_metrics = un_epoched_graph_metrics.sort_values(["subject_ID", "session_ID"]).reset_index(drop=True)

# -----------------------------
# Reporting on missing / unreadable fields (no placeholders in dataframe):
# -----------------------------
if missing_file_pairs:
    print(f"[WARN] {len(missing_file_pairs)} pair(s) missing one/both un-epoched files (row will be NaN-filled):")
    for sub, ses, has_cent, has_glob in missing_file_pairs[:50]:
        print(
            f"  - {sub} | {ses} "
            f"(centrality={'OK' if has_cent else 'MISSING'}, global_metrics={'OK' if has_glob else 'MISSING'})")
    if len(missing_file_pairs) > 50:
        print(f"  ... (showing first 50 of {len(missing_file_pairs)})")

if parse_fail_pairs:
    print(f"[WARN] {len(parse_fail_pairs)} pair(s) failed to load/parse (row will be NaN-filled):")
    for sub, ses, err in parse_fail_pairs[:20]:
        print(f"  - {sub} | {ses} (reason: {err})")
    if len(parse_fail_pairs) > 20:
        print(f"  ... (showing first 20 of {len(parse_fail_pairs)})")

# NOTE: Missing / unreadable file pairs and per-pair parse failures are WARNING-only by design.
#       --> We always proceed & fill missing values w/ NaNs for the affected [subject_ID x session_ID]
if missing_file_pairs or parse_fail_pairs:
    print("[INFO] Proceeding despite missing/unreadable un-epoched inputs (affected rows are NaN-filled).")

# --------------------------------------------------------------------------------
# Prefix non-ID columns for un-epoched session-level data:
# --------------------------------------------------------------------------------

id_columns = {"subject_ID", "session_ID"}

rename_map = {
    col: f"{UN_EPOCHED_DATA_COLUMN_PREFIX}{col}"
    for col in un_epoched_graph_metrics.columns
    if col not in id_columns}

un_epoched_graph_metrics = un_epoched_graph_metrics.rename(columns=rename_map)

print(
    f"[OK] Applied column prefix '{UN_EPOCHED_DATA_COLUMN_PREFIX}' to "
    f"{len(rename_map)} un-epoched data columns.")

# -----------------------------
# Post-checks: duplicate columns / duplicate rows:
# -----------------------------
# Duplicate columns (name collisions):
dup_cols = un_epoched_graph_metrics.columns[un_epoched_graph_metrics.columns.duplicated()].tolist()
if dup_cols:
    print(f"[ERROR] Duplicate column names detected: {len(dup_cols)}")
    # Show unique duplicates + counts:
    vc = pd.Series(dup_cols).value_counts()
    for col, n in vc.head(25).items():
        print(f"  - {col} (duplicated {n+1}x total)")
    if len(vc) > 25:
        print(f"  ... (showing first 25 of {len(vc)})")
    if HARD_STOP:
        raise ValueError("[ERROR] Duplicate column names found in un_epoched_graph_metrics.")
else:
    print("[OK] No duplicate column names detected in un_epoched_graph_metrics.")

# Detect duplicate rows by subject/session:
dup_mask = un_epoched_graph_metrics.duplicated(subset=["subject_ID", "session_ID"], keep=False)
n_dup_rows = int(dup_mask.sum())

if n_dup_rows > 0:
    dup_df = un_epoched_graph_metrics.loc[dup_mask, ["subject_ID", "session_ID"]].copy()
    dup_counts = dup_df.value_counts().reset_index(name="n_rows").sort_values("n_rows", ascending=False)

    print(f"[ERROR] Duplicate subject_ID × session_ID rows detected: {n_dup_rows} row(s) involved.")
    print("[INFO] Top duplicate keys:")
    print(dup_counts.head(20).to_string(index=False))

    if HARD_STOP:
        raise ValueError("[ERROR] Duplicate subject/session rows found in un_epoched_graph_metrics.")
else:
    print("[OK] No duplicate subject_ID × session_ID rows detected in un_epoched_graph_metrics.")

print(f"[OK] un_epoched_graph_metrics built: shape = {un_epoched_graph_metrics.shape}")

Next, we do a very similar thing for the 'EPOCHED' graph metrics data. Once again, the target pair of data tables exist as separate files for each subject_ID x session_ID pair, and are NOT pre-flattened; so some significant unpacking and wrangling is required:

- This step requires the 'EPOCH_LENGH' and 'EPOCH_OVERLAP' parameters to have been set correctly in order to find the target files successfully.
- All output columns from this step are prefixed by the 'TIME_RESOLVED_GRAPH_METRICS_PREFIX' and 'GRAPH_METRICS_ROI_SUMMARIES_PREFIX' substrings, for easy downstream selection/filtering

In [ ]:
# --------------------------------------------------------------------------------
# Build epoched_graph_metrics (graceful NaN-fill, global padding, duplicate checks)
#   - Reads paired epoched outputs from:
#       target_subdir = EPOCHED_DIR / f"window-{EPOCH_LENGTH}_overlap-{EPOCH_OVERLAP}"
#   - Produces one row per subject_ID × session_ID in final_targets
# --------------------------------------------------------------------------------

#########################################################
#########################################################
# Prefixes to append to all output columns from this cell (except subject_ID/session_ID):
TIME_RESOLVED_GRAPH_METRICS_PREFIX   = "GraphMetrics-TimeResolved__"
GRAPH_METRICS_ROI_SUMMARIES_PREFIX   = "GraphMetrics-ROIsummaries__"
#########################################################
#########################################################

# If epoching disabled, stop early:
if not EPOCH_LENGTH:
    print("[INFO] Epoching is currently disabled in config.yaml parameters.")
else:
    # Confirm target subdirectory exists:
    target_subdir = EPOCHED_DIR / f"window-{EPOCH_LENGTH}_overlap-{EPOCH_OVERLAP}"

    if not target_subdir.exists() or not target_subdir.is_dir():
        msg = f"[WARN] Epoched output subdirectory not found: {target_subdir}"
        print(msg)
        # Keep existing behavior (directory missing is still potentially fatal):
        if HARD_STOP:
            raise FileNotFoundError(msg)

    else:
        # ---------------------------------------------------------------------
        # Expected pairs from targets (ONE ROW REQUIRED PER PAIR in output):
        # ---------------------------------------------------------------------
        expected_pairs = sorted(set(zip(final_targets["subject_ID"], final_targets["session_ID"])))

        # Regex patterns for epoched filenames (subject/session extracted; epoch params must match):
        summaries_pat = re.compile(
            rf'^(?P<sub>[^_]+)_(?P<ses>[^_]+)_epoched_window-{re.escape(str(EPOCH_LENGTH))}_overlap-{re.escape(str(EPOCH_OVERLAP))}_summaries\.csv$')
        timecourses_pat = re.compile(
            rf'^(?P<sub>[^_]+)_(?P<ses>[^_]+)_epoched_window-{re.escape(str(EPOCH_LENGTH))}_overlap-{re.escape(str(EPOCH_OVERLAP))}_timecourses\.csv$')

        # Map (subject_ID, session_ID) --> file path:
        summaries_files = {}
        timecourses_files = {}

        for f in target_subdir.glob("*.csv"):
            name = f.name
            m = summaries_pat.match(name)
            if m:
                summaries_files[(m.group("sub"), m.group("ses"))] = f
                continue
            m = timecourses_pat.match(name)
            if m:
                timecourses_files[(m.group("sub"), m.group("ses"))] = f
                continue

        summaries_pairs   = set(summaries_files.keys())
        timecourses_pairs = set(timecourses_files.keys())
        complete_pairs    = summaries_pairs & timecourses_pairs
        missing_pairs     = set(expected_pairs) - complete_pairs

        # Keep original existence-reporting, but WARNING-only (no hard errors):
        if missing_pairs:
            print(
                f"[WARN] {len(missing_pairs)} subject_ID × session_ID pair(s) missing one or more epoched files "
                f"(window={EPOCH_LENGTH}, overlap={EPOCH_OVERLAP}) — rows will be NaN-filled:")
            for sub, ses in sorted(missing_pairs):
                has_sum = (sub, ses) in summaries_pairs
                has_tc  = (sub, ses) in timecourses_pairs
                print(
                    f"  - {sub} | {ses} "
                    f"(summaries={'OK' if has_sum else 'MISSING'}, "
                    f"timecourses={'OK' if has_tc else 'MISSING'})")
        else:
            print(
                "[OK] All subject_ID × session_ID pairs have matching epoched summaries and timecourses files "
                f"(window={EPOCH_LENGTH}, overlap={EPOCH_OVERLAP}).")

        # ----------------------------------------------------------------------
        # Compute GLOBAL padding widths (time_index + roi_label) across dataset:
        #    --> Best-effort; avoids full reads where possible
        # ----------------------------------------------------------------------
        global_max_time = None
        global_max_roi  = None

        # Scan timecourses files (need time_index + roi_label):
        for sub, ses in expected_pairs:
            pth = timecourses_files.get((sub, ses), None)
            if pth is None:
                continue
            try:
                # Read only columns of interest:
                df_tmp = pd.read_csv(pth, usecols=["time_index", "roi_label"])
                t = pd.to_numeric(df_tmp["time_index"], errors="coerce")
                r = pd.to_numeric(df_tmp["roi_label"], errors="coerce")

                if t.notna().any():
                    mx = int(t.dropna().max())
                    global_max_time = mx if global_max_time is None else max(global_max_time, mx)
                if r.notna().any():
                    mx = int(r.dropna().max())
                    global_max_roi = mx if global_max_roi is None else max(global_max_roi, mx)
            except Exception:
                # Best-effort scan only:
                continue

        time_pad = max(1, len(str(int(global_max_time)))) if global_max_time is not None else 1
        roi_pad  = max(1, len(str(int(global_max_roi))))  if global_max_roi  is not None else 1

        if global_max_time is not None:
            print(f"[INFO] Global time_index padding width set to {time_pad} (max time_index observed = {global_max_time}).")
        else:
            print("[WARN] Could not determine global time_index padding width (no readable time_index values). Defaulting to width=1.")

        if global_max_roi is not None:
            print(f"[INFO] Global roi_label padding width set to {roi_pad} (max roi_label observed = {global_max_roi}).")
        else:
            print("[WARN] Could not determine global roi_label padding width (no readable roi_label values). Defaulting to width=1.")

        # ---------------------------------------------------------------------
        # Build rows (one per expected pair); NaN-fill on missing / unreadable:
        # ---------------------------------------------------------------------
        rows = []
        missing_file_pairs = []
        parse_fail_pairs = []

        # Required schemas:
        required_timecourses_cols = {"subject_ID", "session_ID", "roi_label", "metric_name", "time_index", "value"}
        required_summaries_cols   = {"subject_ID", "session_ID", "roi_label", "metric_name", "summary_type", "value"}

        for sub, ses in expected_pairs:
            row = {"subject_ID": sub, "session_ID": ses}

            tc_path  = timecourses_files.get((sub, ses), None)
            sum_path = summaries_files.get((sub, ses), None)

            if tc_path is None or sum_path is None:
                has_tc  = tc_path is not None
                has_sum = sum_path is not None

                print(
                    f"[WARN] Missing epoched inputs for {sub} | {ses} "
                    f"(summaries={'OK' if has_sum else 'MISSING'}, "
                    f"timecourses={'OK' if has_tc else 'MISSING'})")

                missing_file_pairs.append((sub, ses, has_sum, has_tc))
                rows.append(row)
                continue

            try:
                # ----------------------------
                # Load + validate timecourses:
                # ----------------------------
                tc_df = pd.read_csv(tc_path)
                missing_cols = required_timecourses_cols - set(tc_df.columns)
                if missing_cols:
                    raise RuntimeError(f"timecourses missing required columns: {sorted(missing_cols)}")

                # Coerce indices to INTs (& drop rows where coersion fails):
                tc_df = tc_df.copy()
                tc_df["time_index"] = pd.to_numeric(tc_df["time_index"], errors="coerce")
                tc_df["roi_label"]  = pd.to_numeric(tc_df["roi_label"], errors="coerce")
                tc_df = tc_df.loc[tc_df["time_index"].notna() & tc_df["roi_label"].notna()].copy()
                tc_df["time_index"] = tc_df["time_index"].astype(int)
                tc_df["roi_label"]  = tc_df["roi_label"].astype(int)

                # Value is numeric where possible:
                tc_df["value"] = pd.to_numeric(tc_df["value"], errors="coerce")

                # Flatten timecourses --> 't-<time>_ROI-<roi>_<metric>':
                #      --> If duplicates exist, take first non-null; otherwise first
                for _, r in tc_df.iterrows():
                    t = int(r["time_index"])
                    roi = int(r["roi_label"])
                    metric = str(r["metric_name"])
                    col = f"t-{t:0{time_pad}d}_ROI-{roi:0{roi_pad}d}_{metric}"
                    row[f"{TIME_RESOLVED_GRAPH_METRICS_PREFIX}{col}"] = r["value"]

                # --------------------------
                # Load + validate summaries:
                # --------------------------
                sum_df = pd.read_csv(sum_path)
                missing_cols = required_summaries_cols - set(sum_df.columns)
                if missing_cols:
                    raise RuntimeError(f"summaries missing required columns: {sorted(missing_cols)}")

                sum_df = sum_df.copy()
                sum_df["roi_label"] = pd.to_numeric(sum_df["roi_label"], errors="coerce")
                sum_df = sum_df.loc[sum_df["roi_label"].notna()].copy()
                sum_df["roi_label"] = sum_df["roi_label"].astype(int)
                sum_df["value"] = pd.to_numeric(sum_df["value"], errors="coerce")

                # Flatten summaries --> 'ROI-<roi>_<metric>_<summary_type>':
                for _, r in sum_df.iterrows():
                    roi = int(r["roi_label"])
                    metric = str(r["metric_name"])
                    stype = str(r["summary_type"])
                    col = f"ROI-{roi:0{roi_pad}d}_{metric}_{stype}"
                    row[f"{GRAPH_METRICS_ROI_SUMMARIES_PREFIX}{col}"] = r["value"]

            except Exception as e:
                parse_fail_pairs.append((sub, ses, str(e)))
                row = {"subject_ID": sub, "session_ID": ses}

            rows.append(row)

        epoched_graph_metrics = pd.DataFrame(rows)

        # Stable sorting:
        epoched_graph_metrics = epoched_graph_metrics.sort_values(["subject_ID", "session_ID"]).reset_index(drop=True)

        # ---------------------------------------------------------------------
        # Reporting on missing / unreadable (warnings only):
        # ---------------------------------------------------------------------
        if missing_file_pairs:
            print(f"[WARN] {len(missing_file_pairs)} pair(s) missing one/both epoched files (row will be NaN-filled):")
            for sub, ses, has_sum, has_tc in missing_file_pairs[:50]:
                print(
                    f"  - {sub} | {ses} "
                    f"(summaries={'OK' if has_sum else 'MISSING'}, timecourses={'OK' if has_tc else 'MISSING'})")
            if len(missing_file_pairs) > 50:
                print(f"  ... (showing first 50 of {len(missing_file_pairs)})")

        if parse_fail_pairs:
            print(f"[WARN] {len(parse_fail_pairs)} pair(s) failed to load/parse (row will be NaN-filled):")
            for sub, ses, err in parse_fail_pairs[:20]:
                print(f"  - {sub} | {ses} (reason: {err})")
            if len(parse_fail_pairs) > 20:
                print(f"  ... (showing first 20 of {len(parse_fail_pairs)})")

        if missing_file_pairs or parse_fail_pairs:
            print("[INFO] Proceeding despite missing/unreadable epoched inputs (affected rows are NaN-filled).")

        # ---------------------------------------------------------------------
        # Post-checks: duplicate columns / duplicate rows ('HARD_STOP' policy still applies here):
        # ---------------------------------------------------------------------
        dup_cols = epoched_graph_metrics.columns[epoched_graph_metrics.columns.duplicated()].tolist()
        if dup_cols:
            print(f"[ERROR] Duplicate column names detected: {len(dup_cols)}")
            vc = pd.Series(dup_cols).value_counts()
            for col, n in vc.head(25).items():
                print(f"  - {col} (duplicated {n+1}x total)")
            if len(vc) > 25:
                print(f"  ... (showing first 25 of {len(vc)})")
            if HARD_STOP:
                raise ValueError("[ERROR] Duplicate column names found in epoched_graph_metrics.")
        else:
            print("[OK] No duplicate column names detected in epoched_graph_metrics.")

        dup_mask = epoched_graph_metrics.duplicated(subset=["subject_ID", "session_ID"], keep=False)
        n_dup_rows = int(dup_mask.sum())

        if n_dup_rows > 0:
            dup_df = epoched_graph_metrics.loc[dup_mask, ["subject_ID", "session_ID"]].copy()
            dup_counts = dup_df.value_counts().reset_index(name="n_rows").sort_values("n_rows", ascending=False)

            print(f"[ERROR] Duplicate subject_ID × session_ID rows detected: {n_dup_rows} row(s) involved.")
            print("[INFO] Top duplicate keys:")
            print(dup_counts.head(20).to_string(index=False))

            if HARD_STOP:
                raise ValueError("[ERROR] Duplicate subject/session rows found in epoched_graph_metrics.")
        else:
            print("[OK] No duplicate subject_ID × session_ID rows detected in epoched_graph_metrics.")

        print(f"[OK] epoched_graph_metrics built: shape = {epoched_graph_metrics.shape}")

Next, we grab data from the 'HMM_features' output subfamily. These tables have already been pre-formatted / pre-flattened upstream, so we should be able to straightforwardly load in the target data tables and look up each subject_ID / session_ID pair:

- Locating these tables requires the 'DATASET_NAME' variable to have been set correctly

In [ ]:
# --------------------------------------------------------------------------------
# Load & sanity-check: HMM_features flattened table:
# --------------------------------------------------------------------------------

# Use DATASET_NAME to locate correct subfolder:
target_dir = Path(HMM_FEATURES_DIR) / str(DATASET_NAME)

# Verify that directory exists:
if not target_dir.exists() or not target_dir.is_dir():
    msg = f"[WARN] Target directory not found: {target_dir}"
    print(msg)
    if HARD_STOP:
        raise FileNotFoundError(msg)

else:
    flattened_path = target_dir / "_flattened_features_ALL.csv"

    # Verify that file exists:
    if not flattened_path.exists() or not flattened_path.is_file():
        msg = f"[WARN] Required file not found: {flattened_path}"
        print(msg)
        if HARD_STOP:
            raise FileNotFoundError(msg)

    else:
        # Load:
        HMM_features = pd.read_csv(flattened_path)
        print(f"[OK] Loaded HMM_features from: {flattened_path}")
        print(f"     shape = {HMM_features.shape}")

        # Basic schema check (fail fast if malformed):
        required_cols = {"subject_ID", "session_ID"}
        missing_cols = required_cols - set(HMM_features.columns)
        if missing_cols:
            msg = f"[ERROR] HMM_features missing required columns: {sorted(missing_cols)}"
            print(msg)
            if HARD_STOP:
                raise ValueError(msg)

        # Coverage check against final_targets [subject_ID x session_ID]:
        expected_pairs = set(zip(final_targets["subject_ID"], final_targets["session_ID"]))
        observed_pairs = set(zip(HMM_features["subject_ID"], HMM_features["session_ID"]))

        missing_pairs = expected_pairs - observed_pairs

        if missing_pairs:
            print(
                f"[ERROR] {len(missing_pairs)} subject_ID x session_ID pair(s) from final_targets "
                "missing in HMM_features:")
            for sub, ses in sorted(missing_pairs):
                print(f"  - {sub} | {ses}")
            if HARD_STOP:
                raise ValueError(
                    "[ERROR] One or more subject/session pairs in final_targets are missing from HMM_features.")
        else:
            print("[OK] HMM_features contains a matching row for every subject_ID x session_ID in final_targets.")

Finally, similar checks for tables within the 'HMM_states' directory -- once again these should have been pre-formatted / pre-flattened upstream:

In [ ]:
# --------------------------------------------------------------------------------
# Load & sanity-check: HMM_states flattened table:
# --------------------------------------------------------------------------------

# Use 'DATASET_NAME' to locate correct subfolder:
target_dir = Path(HMM_STATES_DIR) / str(DATASET_NAME)

# Verify that directory exists:
if not target_dir.exists() or not target_dir.is_dir():
    msg = f"[WARN] Target directory not found: {target_dir}"
    print(msg)
    if HARD_STOP:
        raise FileNotFoundError(msg)

else:
    flattened_path = target_dir / "_flattened_HMM_state_features_ALL.csv"

    # Verify that file exists:
    if not flattened_path.exists() or not flattened_path.is_file():
        msg = f"[WARN] Required file not found: {flattened_path}"
        print(msg)
        if HARD_STOP:
            raise FileNotFoundError(msg)

    else:
        # Load:
        HMM_states = pd.read_csv(flattened_path)
        print(f"[OK] Loaded HMM_states from: {flattened_path}")
        print(f"     shape = {HMM_states.shape}")

        # Basic schema check (fail fast if malformed):
        required_cols = {"subject_ID", "session_ID"}
        missing_cols = required_cols - set(HMM_states.columns)
        if missing_cols:
            msg = f"[ERROR] HMM_states missing required columns: {sorted(missing_cols)}"
            print(msg)
            if HARD_STOP:
                raise ValueError(msg)

        # Coverage check against final_targets [subject_ID x session_ID]:
        expected_pairs = set(zip(final_targets["subject_ID"], final_targets["session_ID"]))
        observed_pairs = set(zip(HMM_states["subject_ID"], HMM_states["session_ID"]))

        missing_pairs = expected_pairs - observed_pairs

        if missing_pairs:
            print(
                f"[ERROR] {len(missing_pairs)} subject_ID x session_ID pair(s) from final_targets "
                "missing in HMM_states:")
            for sub, ses in sorted(missing_pairs):
                print(f"  - {sub} | {ses}")
            if HARD_STOP:
                raise ValueError(
                    "[ERROR] One or more subject/session pairs in final_targets are missing from HMM_states.")
        else:
            print("[OK] HMM_states contains a matching row for every subject_ID x session_ID in final_targets.")

-----------
#### All data should now have been gathered, now we just need to join it all together:

One quick preliminary step: let's prefix all the HMM-derived table columns with a family prefix, for easy feature-grouping & -filtering:

In [ ]:
# ------------------------------------------------------------
# Add HMM table prefixes:
# ------------------------------------------------------------

#########################################################
#########################################################
HMM_FEATURES_PREFIX = "HMM__"
HMM_STATES_PREFIX   = "HMM__"
#########################################################
#########################################################

id_columns = {"subject_ID", "session_ID"}
# Prefix 'HMM_features' table columns:
HMM_features = HMM_features.rename(
    columns={
        c: f"{HMM_FEATURES_PREFIX}{c}"
        for c in HMM_features.columns
        if c not in id_columns})
# Prefix 'HMM_states' table columns:
HMM_states = HMM_states.rename(
    columns={
        c: f"{HMM_STATES_PREFIX}{c}"
        for c in HMM_states.columns
        if c not in id_columns})
print(f"[OK] Prefixed {len(HMM_features.columns) - len(id_columns)} columns in HMM_features.")
print(f"[OK] Prefixed {len(HMM_states.columns) - len(id_columns)} columns in HMM_states.")

Quick preview of all possible tables (may individually be empty if not toggled / required data is missing):

In [ ]:
display(final_targets.head())
display(HMM_features.head())
display(HMM_states.head())
display(un_epoched_graph_metrics.head())
display(anatomy_table.head())
display(epoched_graph_metrics.head())

**Final (conditional) join onto master 'final_df' dataframe:**

In [ ]:
# -----------------------------------------------------------------------------------
# FINAL JOIN: build ML-ready 'final_df' from final_targets + selected feature blocks:
# -----------------------------------------------------------------------------------

final_df = final_targets.copy()
session_join_keys = ["subject_ID", "session_ID"]
subject_join_key  = ["subject_ID"]

def _safe_left_merge_session_level(
    base_df: pd.DataFrame, add_df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Left-merge session-level table on subject_ID x session_ID."""
    missing_in_base = set(session_join_keys) - set(base_df.columns)
    missing_in_add  = set(session_join_keys) - set(add_df.columns)
    if missing_in_base:
        raise RuntimeError(f"[JOIN ERROR] final_df missing join key(s): {sorted(missing_in_base)}")
    if missing_in_add:
        raise RuntimeError(f"[JOIN ERROR] {name} missing join key(s): {sorted(missing_in_add)}")
    # Prevent row multiplication:
    dup_mask = add_df.duplicated(subset=session_join_keys, keep=False)
    if dup_mask.any():
        dup_counts = (
            add_df.loc[dup_mask, session_join_keys]
            .value_counts()
            .reset_index(name="n_rows")
            .sort_values("n_rows", ascending=False))
        raise RuntimeError(
            f"[JOIN ERROR] {name} has duplicate subject_ID x session_ID keys.\n"
            f"Top duplicates:\n{dup_counts.head(20).to_string(index=False)}")
    out = base_df.merge(add_df, on=session_join_keys, how="left", validate="one_to_one")
    print(f"[OK] Joined {name} (session-level): final_df shape = {out.shape}")
    return out

def _safe_left_merge_subject_level(
    base_df: pd.DataFrame, add_df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Left-merge subject-level table on subject_ID only."""
    if "subject_ID" not in base_df.columns:
        raise RuntimeError("[JOIN ERROR] final_df missing 'subject_ID'")
    if "subject_ID" not in add_df.columns:
        raise RuntimeError(f"[JOIN ERROR] {name} missing 'subject_ID'")
    # Prevent row multiplication across sessions:
    dup_mask = add_df.duplicated(subset=["subject_ID"], keep=False)
    if dup_mask.any():
        dup_counts = (
            add_df.loc[dup_mask, ["subject_ID"]]
            .value_counts()
            .reset_index(name="n_rows")
            .sort_values("n_rows", ascending=False))
        raise RuntimeError(
            f"[JOIN ERROR] {name} has duplicate subject_ID rows.\n"
            f"Top duplicates:\n{dup_counts.head(20).to_string(index=False)}")
    out = base_df.merge(add_df, on="subject_ID", how="left", validate="many_to_one")
    print(f"[OK] Joined {name} (subject-level): final_df shape = {out.shape}")
    return out

# ------------------------------------------------------------------------------
# Conditional joins (in order):
# ------------------------------------------------------------------------------
if OUTPUT_HMM_FEATURES:
    final_df = _safe_left_merge_session_level(final_df, HMM_features, "HMM_features")
if OUTPUT_HMM_STATES:
    final_df = _safe_left_merge_session_level(final_df, HMM_states, "HMM_states")
if OUTPUT_GRAPH_METRICS_SUMMARIES:
    final_df = _safe_left_merge_session_level(
        final_df, un_epoched_graph_metrics, "un_epoched_graph_metrics")
if OUTPUT_GRAPH_METRICS_TIME_RESOLVED:
    final_df = _safe_left_merge_session_level(
        final_df, epoched_graph_metrics, "epoched_graph_metrics")
if OUTPUT_ANATOMY_MEASURES:
    final_df = _safe_left_merge_subject_level(final_df, anatomy_table, "anatomy_table")

print(f"\n[OK] final_df assembled: shape = {final_df.shape}")

Review:

In [ ]:
final_df.sample(25)

Final save / export:

In [ ]:
# Resolve output directory:
if FINAL_OUTPUT_DIR:
    output_dir = Path(BASE_DIRECTORY) / FINAL_OUTPUT_DIR
else:
    output_dir = Path(BASE_DIRECTORY)
output_dir.mkdir(parents=True, exist_ok=True)

# Set path:
output_path = output_dir / "master_output.csv"

# Write to disk:
final_df.to_csv(output_path, index=False)

print(f"[OK] Final master output written to: {output_path}")
print(f"     shape = {final_df.shape}")

---------